# IndicSpeak — Inference Walkthrough

Orpheus-style TTS (Llama-3.2-3B backbone → SNAC 24 kHz codec tokens) through the public engine API:

| mode | offline (`IndicTTSEngine`) | streaming (`IndicStreamingTTSEngine`) |
|---|---|---|
| single utterance | `synthesize` / `synthesize_batch` | `stream` / `stream_sync` |
| conversation (message list) | `synthesize_conversation` | `stream_conversation` / `stream_conversation_sync` |

**Prerequisites**

- A GPU node with the environment installed: `./install.sh && source .venv/bin/activate`
  (see the README for the cu129 torch → flash-attn → package install order).
- A model checkpoint. `canopylabs/orpheus-3b-0.1-pretrained` works for smoke-testing the
  pipeline; for real speech use your own SFT checkpoint from `notebooks/training.ipynb`.
- ⚠️ **Vocab trap**: load the checkpoint's *own* tokenizer (the default) unless the checkpoint
  was trained against the extended 156942-token tokenizer. Mixing them silently corrupts
  speaker/style ids — see the README troubleshooting section.

In [ ]:
import torch

import bodhan_genai.tts

print("bodhan-genai:", bodhan_genai.tts.__version__)
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "inference needs a GPU node"
print("GPUs:", torch.cuda.device_count(), torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

# --- edit these ------------------------------------------------------------
MODEL = "canopylabs/orpheus-3b-0.1-pretrained"  # or /path/to/your/sft/checkpoint
TOKENIZER = None  # None = the checkpoint's own tokenizer (safe default)
SPEAKER = "S1"  # speaker id the checkpoint was trained with ("" = none)
OUT_DIR = Path("notebook_out")
OUT_DIR.mkdir(exist_ok=True)
# ----------------------------------------------------------------------------

## 1. Offline synthesis

`IndicTTSEngine` loads everything once (vLLM engine by default + a resident SNAC decoder) and
synthesizes on demand. Construction takes a minute the first time (weights + CUDA graphs); every
`synthesize` after that is fast.

In [ ]:
from bodhan_genai.tts import IndicTTSEngine

engine = IndicTTSEngine(
    MODEL,
    tokenizer=TOKENIZER,
    # backend="vllm" is the default; see §5 for the HF backend.
    gpu_memory_utilization=0.85,
    max_model_len=8192,
)

In [ ]:
from IPython.display import Audio, display

result = engine.synthesize("Hello! This is IndicSpeak speaking.", speaker=SPEAKER)
print(
    f"{result.duration_s:.2f}s audio | prompt {result.prompt_tokens} tok | "
    f"audio {result.audio_tokens} tok | gen {result.gen_time_s:.2f}s | "
    f"decode {result.decode_time_s:.2f}s | RTF {result.rtf:.2f}"
)
result.save(OUT_DIR / "hello.wav")
display(Audio(result.audio, rate=result.sample_rate))

## 2. Sampling control

Defaults come from `SamplingConfig` (temperature 0.6, top_p 0.95, top_k −1, repetition_penalty
1.1, max_new_tokens 2048 ≈ 24 s of audio). Set engine-wide defaults at construction
(`sampling=SamplingConfig(...)`) or override per call — `None` kwargs fall through to the
engine's defaults.

In [ ]:
from bodhan_genai.tts import SamplingConfig

print("canonical defaults:", SamplingConfig())

flat = engine.synthesize(
    "The measurement is exactly five point two units.", speaker=SPEAKER, temperature=0.0
)  # deterministic
expressive = engine.synthesize(
    "No way — that's incredible news!!", speaker=SPEAKER, temperature=0.8, top_p=0.9
)
display(Audio(flat.audio, rate=flat.sample_rate))
display(Audio(expressive.audio, rate=expressive.sample_rate))

## 3. Batch synthesis

`synthesize_batch` sends every prompt through **one** vLLM generate call and one batched SNAC
decode. A failed row comes back with `result.error` set instead of sinking the batch.

In [ ]:
texts = [
    "First sentence of the batch.",
    "Second one, a little longer, to show length variety in a single call.",
    "",  # deliberately blank -> comes back with .error set
    "And a final fourth sentence.",
]
results = engine.synthesize_batch(texts, speakers=SPEAKER)
for i, r in enumerate(results):
    if r.error:
        print(f"[{i}] FAILED: {r.error}")
    else:
        path = r.save(OUT_DIR / f"batch_{i}.wav")
        print(f"[{i}] {r.duration_s:5.2f}s -> {path}")

## 4. Conversation synthesis

The second synthesis mode: pass a whole conversation as a chat-style message list. Turns are
rendered through the **conversation template** — `<|speaker>NAME<speaker|>` tags inline in the
text, no per-utterance metadata prefix — and the model produces **one continuous multi-speaker
audio sample**, exactly the serialization it saw for conversation rows in training.

In [ ]:
messages = [
    {"speaker": "S1", "text": "Hey, did you get a chance to look at the draft?"},
    {"speaker": "S2", "text": "I did! Left a few comments, mostly about the intro."},
    {"speaker": "S1", "text": "Perfect, I'll pick them up this afternoon."},
]
convo = engine.synthesize_conversation(messages)
print(f"{convo.duration_s:.2f}s conversation, {convo.audio_tokens} audio tokens")
convo.save(OUT_DIR / "dialogue.wav")
display(Audio(convo.audio, rate=convo.sample_rate))

## 5. HF backend (PEFT adapters / debugging)

`backend="hf"` swaps vLLM for plain `AutoModelForCausalLM.generate` — slower for batches, but it
loads LoRA adapters directly via `adapter_dir` and is easier to step through. Close the vLLM
engine first: **one engine per process** is the rule (both want most of the GPU).

In [ ]:
engine.close()  # free the vLLM engine before loading another backend

hf_engine = IndicTTSEngine(
    MODEL,
    backend="hf",
    tokenizer=TOKENIZER,
    # adapter_dir="/path/to/lora/adapter",   # <- LoRA checkpoint from train_lora.sh
)
hf_engine.synthesize("Testing the HuggingFace backend.", speaker=SPEAKER).save(
    OUT_DIR / "hf_backend.wav"
)
hf_engine.close()

## 6. Streaming

`IndicStreamingTTSEngine` is the same core the production websocket server runs on: vLLM
AsyncLLM → sliding-window SNAC decode → raw **int16 PCM frames at 24 kHz** as generation
progresses. In a notebook, use the `*_sync` wrappers (they drive a private event loop); in an
async service, use `stream(...)` / `stream_conversation(...)` directly.

In [ ]:
import time

import numpy as np

from bodhan_genai.tts import IndicStreamingTTSEngine

stream_engine = IndicStreamingTTSEngine(MODEL, tokenizer=TOKENIZER)

t0, first, chunks = time.perf_counter(), None, []
for pcm in stream_engine.stream_sync(
    "Streaming synthesis begins playing long before the sentence is finished.",
    speaker=SPEAKER,
    frames_per_message=4,
):
    if first is None:
        first = time.perf_counter() - t0
    chunks.append(pcm)

audio = np.frombuffer(b"".join(chunks), dtype=np.int16).astype(np.float32) / 32767.0
print(f"time-to-first-audio {first * 1000:.0f} ms | {len(audio) / 24_000:.2f}s total")
display(Audio(audio, rate=24_000))

In [ ]:
import asyncio

# Streaming a conversation works the same way:
pcm_parts = list(stream_engine.stream_conversation_sync(messages, frames_per_message=4))
convo_audio = np.frombuffer(b"".join(pcm_parts), dtype=np.int16).astype(np.float32) / 32767.0
display(Audio(convo_audio, rate=24_000))

asyncio.run(stream_engine.shutdown())  # stops the AsyncLLM + SNAC micro-batcher

## Where to go next

- **Bulk manifests** (thousands of rows, multi-GPU): `scripts/tts/infer.sh --jsonl-path eval.jsonl
  --output_dir out/` — the two-phase Ray pipeline, config in `configs/tts/infer/offline_vllm.yaml`.
- **Production serving**: `CHECKPOINT=... scripts/tts/serve.sh` starts the Ray Serve server (one
  replica per GPU) with three endpoints — WS `/tts`, WS `/tts/chunked`, POST `/tts/offline`;
  `python examples/tts/streaming_client.py --mode {stream,chunked,offline} --text "..."` talks to it.
  Details in [docs/tts/serving.md](../../docs/tts/serving.md).
- **Train your own checkpoint**: [notebooks/training.ipynb](training.ipynb).